In [8]:
# ==============================================================================
# BLOCK 1: IMPORTS AND PGN MENTOR GRANDMASTER DATASET DOWNLOAD
# ==============================================================================
!pip install -q python-chess requests tqdm

import io
import time
import zipfile
import requests
import chess
import chess.pgn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

print(f"[INFO] Initializing BOZON v0.3 ResNet Backbone")
print(f"[INFO] Computing Device: {device}")

PGN_MENTOR_PLAYERS = [
    "Carlsen",
    "Kasparov",
    "Fischer",
    "Tal",
    "Karpov",
    "Anand"
]

def fetch_pgn_mentor_corpus(players: list) -> str:
    """Downloads and extracts player ZIP archives from pgnmentor.com."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
    }
    combined_pgn = []

    for player in players:
        url = f"https://www.pgnmentor.com/players/{player}.zip"
        print(f"[INFO] Downloading {player}.zip...")
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()

            with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
                for filename in zf.namelist():
                    if filename.endswith(".pgn"):
                        pgn_text = zf.read(filename).decode("utf-8", errors="ignore")
                        combined_pgn.append(pgn_text)
                        print(f"[SUCCESS] Extracted '{filename}' ({len(pgn_text) // 1024:,} KB)")
        except Exception as exc:
            print(f"[WARNING] Failed to fetch {player}: {exc}")

    full_corpus = "\n\n".join(combined_pgn)
    print(f"[INFO] Total Corpus Size: {len(full_corpus) / (1024 * 1024):.2f} MB")
    return full_corpus


pgn_data = fetch_pgn_mentor_corpus(PGN_MENTOR_PLAYERS)

[INFO] Initializing BOZON v0.3 ResNet Backbone
[INFO] Computing Device: cuda
[INFO] Downloading Carlsen.zip...
[SUCCESS] Extracted 'Carlsen.pgn' (5,598 KB)
[INFO] Downloading Kasparov.zip...
[SUCCESS] Extracted 'Kasparov.pgn' (1,318 KB)
[INFO] Downloading Fischer.zip...
[SUCCESS] Extracted 'Fischer.pgn' (537 KB)
[INFO] Downloading Tal.zip...
[SUCCESS] Extracted 'Tal.pgn' (1,421 KB)
[INFO] Downloading Karpov.zip...
[SUCCESS] Extracted 'Karpov.pgn' (2,330 KB)
[INFO] Downloading Anand.zip...
[SUCCESS] Extracted 'Anand.pgn' (2,780 KB)
[INFO] Total Corpus Size: 13.66 MB


In [9]:
# ==============================================================================
# BLOCK 2: BOARD ENCODING AND 100K DATASET EXTRACTION
# ==============================================================================
def board_to_tensor(board: chess.Board) -> np.ndarray:
    """Converts a chess.Board into a (14, 8, 8) float32 tensor."""
    tensor = np.zeros((14, 8, 8), dtype=np.float32)
    for square, piece in board.piece_map().items():
        row = 7 - (square // 8)
        col = square % 8
        channel = piece.piece_type - 1 + (6 if piece.color == chess.BLACK else 0)
        tensor[channel, row, col] = 1.0

    if board.turn == chess.WHITE:
        tensor[12, :, :] = 1.0

    for move in board.legal_moves:
        to_sq = move.to_square
        tensor[13, 7 - (to_sq // 8), to_sq % 8] = 1.0

    return tensor


def encode_move(move: chess.Move) -> int:
    return move.from_square * 64 + move.to_square


class BozonChessDataset(Dataset):
    def __init__(self, pgn_text: str, max_positions: int = 100000):
        self.states = np.zeros((max_positions, 14, 8, 8), dtype=np.float32)
        self.labels = np.zeros((max_positions,), dtype=np.int64)

        pgn_io = io.StringIO(pgn_text)
        games_count = 0
        pos_idx = 0

        print(f"[INFO] Extracting up to {max_positions:,} positions...")

        while pos_idx < max_positions:
            game = chess.pgn.read_game(pgn_io)
            if game is None:
                break
            games_count += 1
            board = game.board()

            for move in game.mainline_moves():
                self.states[pos_idx] = board_to_tensor(board)
                self.labels[pos_idx] = encode_move(move)
                board.push(move)
                pos_idx += 1

                if pos_idx % 25000 == 0:
                    print(f"[PROGRESS] Extracted {pos_idx:,} / {max_positions:,} positions...")

                if pos_idx >= max_positions:
                    break

        self.states = torch.from_numpy(self.states[:pos_idx])
        self.labels = torch.from_numpy(self.labels[:pos_idx])
        print(f"[SUCCESS] Parsed {games_count:,} games -> Total positions: {len(self.states):,}")

    def __len__(self) -> int:
        return len(self.states)

    def __getitem__(self, idx: int):
        return self.states[idx], self.labels[idx]


full_dataset = BozonChessDataset(pgn_data, max_positions=100000)
del pgn_data

train_size = int(0.90 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

print(f"[INFO] Train Samples: {len(train_dataset):,} | Validation Samples: {len(val_dataset):,}")

[INFO] Extracting up to 100,000 positions...
[PROGRESS] Extracted 25,000 / 100,000 positions...
[PROGRESS] Extracted 50,000 / 100,000 positions...
[PROGRESS] Extracted 75,000 / 100,000 positions...
[PROGRESS] Extracted 100,000 / 100,000 positions...
[SUCCESS] Parsed 1,149 games -> Total positions: 100,000
[INFO] Train Samples: 90,000 | Validation Samples: 10,000


In [10]:
# ==============================================================================
# BLOCK 3: RESIDUAL BLOCK AND BOZON v0.3 RESNET ARCHITECTURE
# ==============================================================================
class ResidualBlock(nn.Module):
    """
    Standard AlphaZero-style Residual Block with skip connection.
    Maintains spatial resolution (8x8) while deepening feature extraction.
    """
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)


class BozonResNet(nn.Module):
    """
    BOZON v0.3 Deep Residual Policy Network.
    Uses a 6-block ResNet tower followed by a bottleneck Policy Head
    to prevent dense-layer overfitting.
    """
    def __init__(self, num_blocks: int = 6, channels: int = 128):
        super().__init__()
        # Initial stem convolution
        self.stem = nn.Sequential(
            nn.Conv2d(14, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU()
        )

        # Residual Tower
        self.res_tower = nn.Sequential(
            *[ResidualBlock(channels) for _ in range(num_blocks)]
        )

        # Bottleneck Policy Head (reduces 128 channels -> 32 before FC)
        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Dropout(p=0.25),
            nn.Linear(32 * 8 * 8, 4096)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.res_tower(x)
        logits = self.policy_head(x)
        return logits


def evaluate_model(model: nn.Module, dataloader: DataLoader):
    """Computes Validation Loss, Top-1, and Top-3 Accuracy."""
    model.eval()
    val_loss = 0.0
    top1_correct = 0
    top3_correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for states, labels in dataloader:
            states, labels = states.to(device), labels.to(device)
            logits = model(states)
            loss = criterion(logits, labels)
            val_loss += loss.item() * states.size(0)

            _, top3_preds = torch.topk(logits, k=3, dim=1)
            top1_correct += (top3_preds[:, 0] == labels).sum().item()
            top3_correct += (top3_preds == labels.unsqueeze(1)).any(dim=1).sum().item()
            total += states.size(0)

    return val_loss / total, (top1_correct / total) * 100, (top3_correct / total) * 100


model = BozonResNet(num_blocks=6, channels=128).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"[INFO] BozonResNet (6 Residual Blocks) initialized.")
print(f"[DEBUG] Trainable Parameters: {total_params:,} (Compact & Deep)")

[INFO] BozonResNet (6 Residual Blocks) initialized.
[DEBUG] Trainable Parameters: 10,185,792 (Compact & Deep)


In [11]:
# ==============================================================================
# BLOCK 4: TRAINING LOOP WITH LABEL SMOOTHING AND BEST MODEL CHECKPOINTING
# ==============================================================================
# Label smoothing prevents overconfident logits and reduces validation loss drift
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=2e-4)
epochs = 15
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_val_top1 = 0.0
weights_path = "bozon_v0.3_resnet.pt"

print("[INFO] Starting BOZON v0.3 ResNet training...")

for epoch in range(1, epochs + 1):
    start_t = time.time()
    model.train()
    train_loss = 0.0
    total_train = 0

    for states, labels in train_loader:
        states, labels = states.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(states)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * states.size(0)
        total_train += states.size(0)

    scheduler.step()
    avg_train_loss = train_loss / total_train
    val_loss, val_top1, val_top3 = evaluate_model(model, val_loader)
    elapsed = time.time() - start_t

    # Save checkpoint only when Validation Top-1 improves
    saved_marker = ""
    if val_top1 > best_val_top1:
        best_val_top1 = val_top1
        torch.save(model.state_dict(), weights_path)
        saved_marker = " [BEST SAVED]"

    print(
        f"[EPOCH {epoch:02d}/{epochs:02d}] ({elapsed:.1f}s) | "
        f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val Top-1: {val_top1:.2f}% | Val Top-3: {val_top3:.2f}%{saved_marker}"
    )

# Reload best checkpoint to verify checksum
model.load_state_dict(torch.load(weights_path, map_location=device))
weight_checksum = float(next(model.parameters()).sum().item())

print(f"\n[SUCCESS] Best v0.3 ResNet weights saved to '{weights_path}'")
print(f"[METRIC] Best Validation Top-1 Accuracy: {best_val_top1:.2f}%")
print(f"[VERIFICATION] First Layer Weight Sum Checksum: {weight_checksum:.6f}")

[INFO] Starting BOZON v0.3 ResNet training...
[EPOCH 01/15] (14.5s) | Train Loss: 5.2384 | Val Loss: 3.7389 | Val Top-1: 19.88% | Val Top-3: 37.00% [BEST SAVED]
[EPOCH 02/15] (13.7s) | Train Loss: 3.4601 | Val Loss: 3.3865 | Val Top-1: 23.50% | Val Top-3: 41.44% [BEST SAVED]
[EPOCH 03/15] (13.9s) | Train Loss: 2.7574 | Val Loss: 3.3254 | Val Top-1: 25.20% | Val Top-3: 44.09% [BEST SAVED]
[EPOCH 04/15] (14.0s) | Train Loss: 2.2970 | Val Loss: 3.4564 | Val Top-1: 25.88% | Val Top-3: 44.38% [BEST SAVED]
[EPOCH 05/15] (14.2s) | Train Loss: 1.9610 | Val Loss: 3.5653 | Val Top-1: 25.94% | Val Top-3: 44.59% [BEST SAVED]
[EPOCH 06/15] (14.2s) | Train Loss: 1.7213 | Val Loss: 3.6848 | Val Top-1: 27.06% | Val Top-3: 44.99% [BEST SAVED]
[EPOCH 07/15] (14.1s) | Train Loss: 1.5388 | Val Loss: 3.8160 | Val Top-1: 27.04% | Val Top-3: 45.31%
[EPOCH 08/15] (14.1s) | Train Loss: 1.3931 | Val Loss: 3.8733 | Val Top-1: 26.84% | Val Top-3: 44.80%
[EPOCH 09/15] (14.1s) | Train Loss: 1.2841 | Val Loss: 3.892

In [12]:
# ==============================================================================
# BLOCK 5: INFERENCE SANITY CHECK ON GRANDMASTER OPENING
# ==============================================================================
def predict_next_move(board: chess.Board, model: nn.Module, top_k: int = 3):
    model.eval()
    state_tensor = torch.tensor(
        board_to_tensor(board), dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(state_tensor).squeeze(0)

    legal_mask = torch.full((4096,), float("-inf"), device=device)
    legal_moves_dict = {}

    for move in board.legal_moves:
        idx = encode_move(move)
        legal_mask[idx] = 0.0
        if idx not in legal_moves_dict or move.promotion == chess.QUEEN:
            legal_moves_dict[idx] = move

    probs = torch.softmax(logits + legal_mask, dim=0)
    k = min(top_k, len(legal_moves_dict))
    top_probs, top_indices = torch.topk(probs, k)

    return [
        (legal_moves_dict[idx.item()].uci(), board.san(legal_moves_dict[idx.item()]), prob.item() * 100)
        for prob, idx in zip(top_probs, top_indices)
    ]


# Test on Queen's Gambit Declined position (1. d4 d5 2. c4 e6)
test_board = chess.Board()
for m in ["d2d4", "d7d5", "c2c4", "e7e6"]:
    test_board.push_uci(m)

print("[INFO] Testing BOZON v0.3 on Queen's Gambit Declined (1. d4 d5 2. c4 e6):")
for rank, (uci, san, conf) in enumerate(predict_next_move(test_board, model), 1):
    print(f"Rank #{rank}: {san:<5} ({uci}) | Confidence: {conf:.2f}%")

[INFO] Testing BOZON v0.3 on Queen's Gambit Declined (1. d4 d5 2. c4 e6):
Rank #1: Nf3   (g1f3) | Confidence: 60.88%
Rank #2: Nc3   (b1c3) | Confidence: 34.10%
Rank #3: g3    (g2g3) | Confidence: 2.29%
